In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.5)

from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Dict, List
from pydantic import BaseModel, Field

### Pydantic LLM schema
The list of tasks will be created using the Pydantic LLM schema

In [5]:
class llm_schema(BaseModel):
    task: List[str] = Field(description="A list of tasks to be performed by the LLM. Each task is a string describing a specific question or instruction that the LLM should address.")

llm_with_schema = llm.with_structured_output(llm_schema)

llm_with_schema.invoke("what is the capital of India and who is the prime minister of europe?")

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 42.991062921s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

### Graph Schema

In [6]:
class state_schema(TypedDict):
    tasks: List[str]
    query: str
    results: List[str]
    summary: str

In [8]:
def orchestrator_node(state:state_schema)-> state_schema:
    user_input = state["query"]

    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an assistant that breaks down complex queries into simpler tasks."),
        ("user", f"{user_input}, please generate a list of specific tasks that need to be performed to answer the query comprehensively.")
    ])

    # create a chain to invoke the LLM with the prompt and the defined schema
    chain = prompt | llm_with_schema
    response = chain.invoke({"query": user_input})
    state["tasks"] = response.task
    return state

In [9]:
user_input = "What is the capital of India and who is the prime minister of Europe?"
prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an assistant that breaks down complex queries into simpler tasks."),
        ("user", f"{user_input}, please generate a list of specific tasks that need to be performed to answer the query comprehensively.")
    ])

# create a chain to invoke the LLM with the prompt and the defined schema
chain = prompt | llm_with_schema
response = chain.invoke({"query": user_input})
response

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 16.981311316s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT